# Task 2. Exploratory data analysis and time series characterisation

All seven mandated EDA items: distribution, multi-area preview, rolling stats with
stationarity tests, differencing, STL decomposition with strength scores, ACF/PACF,
spatial heatmap, and anomalies. Each cell imports a function from `mtraffic.eda` and
renders the resulting figure or summary table.

Run `make ingest` first.


In [ ]:
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Image, display

from mtraffic.config import Config
from mtraffic.data.loaders import area_totals, load_area_series
from mtraffic.eda import (
    anomalies,
    correlation,
    decomposition,
    distributions,
    spatial,
    stationarity,
    timeseries,
)

cfg = Config.load()
INTERIM = cfg.paths.interim_dir
FIG = cfg.paths.reports_dir / 'figures'
FIG.mkdir(parents=True, exist_ok=True)

# Focus area: square 5161 (the city-centre top area).
AREA = 5161
series = load_area_series(INTERIM, AREA)
print(f'Area {AREA}: {len(series):,} observations from {series.index.min()} to {series.index.max()}')

## I. City-wide distribution


In [ ]:
totals = area_totals(INTERIM)
stats = distributions.plot_city_pdf(totals, FIG / 'city_pdf.png', annotate_areas=[5161, 4159, 4556])
stats

In [ ]:
display(Image(FIG / 'city_pdf.png'))

## II. Three areas, first two weeks


In [ ]:
timeseries.plot_three_areas_two_weeks(
    INTERIM,
    [5161, 4159, 4556],
    FIG / 'three_area_2weeks.png',
    start=datetime(2013, 11, 1),
    days=14,
    labels=['top (5161)', '4159', '4556'],
)
display(Image(FIG / 'three_area_2weeks.png'))

## III. Rolling mean and standard deviation, ADF and KPSS


In [ ]:
rolling_stats = stationarity.plot_rolling(
    series,
    window=cfg.eda.rolling_window_steps,
    out_png=FIG / f'rolling_{AREA}.png',
    title=f'Rolling mean and std, area {AREA}',
)
rolling_stats

In [ ]:
display(Image(FIG / f'rolling_{AREA}.png'))

## IV. Differencing


In [ ]:
diff1 = stationarity.plot_diff(series, lag=1, out_png=FIG / f'diff1_{AREA}.png', title='First difference')
diff144 = stationarity.plot_diff(series, lag=144, out_png=FIG / f'diff144_{AREA}.png', title='Seasonal difference, lag 144')
{'first': diff1, 'seasonal_144': diff144}

## V. STL decomposition with seasonal and trend strength


In [ ]:
dec_daily = decomposition.stl_decompose(series, period=144)
strengths_daily = decomposition.strengths(dec_daily)
decomposition.plot_decomposition(dec_daily, FIG / f'decompose_{AREA}_daily.png', f'STL daily, area {AREA}')
strengths_daily

In [ ]:
display(Image(FIG / f'decompose_{AREA}_daily.png'))

## VI. ACF and PACF


In [ ]:
acf_summary = correlation.plot_acf_pacf(
    series,
    out_acf=FIG / f'acf_{AREA}.png',
    out_pacf=FIG / f'pacf_{AREA}.png',
    acf_lags=cfg.eda.acf_max_lag,
    pacf_lags=cfg.eda.pacf_max_lag,
    title_prefix=f'area {AREA}:',
)
acf_summary

In [ ]:
display(Image(FIG / f'acf_{AREA}.png'))
display(Image(FIG / f'pacf_{AREA}.png'))

## VII. Spatial distribution


In [ ]:
spatial_stats = spatial.plot_spatial(totals, FIG / 'spatial_heatmap.png')
spatial_stats

In [ ]:
display(Image(FIG / 'spatial_heatmap.png'))

## VIII. Anomaly detection


In [ ]:
outliers = anomalies.stl_residual_outliers(series, period=144)
drops = anomalies.seasonal_naive_drops(series)
anomalies.plot_anomalies(series, outliers, drops, FIG / f'anomalies_{AREA}.png', f'Anomalies, area {AREA}')
print(f'STL residual outliers: {len(outliers)}')
print(f'Seasonal-naive drop runs: {len(drops)}')
outliers.head()

In [ ]:
display(Image(FIG / f'anomalies_{AREA}.png'))